In [ ]:
# Install required libraries (if not already installed)
!pip install transformers datasets scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import os, json, random, shutil
from sklearn.model_selection import train_test_split

INPUT_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/final_datasets/summarization_ready.jsonl"
OUTPUT_DIR  = "/content/drive/MyDrive/biomedical_text_generation/data/tokenized"
UNSEEN_PATH = "/content/drive/MyDrive/biomedical_text_generation/data/unseen"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(UNSEEN_PATH, exist_ok=True)

# Fixed padding lengths (stable)
MAX_INPUT_LENGTH  = 1024 #512 for t5 style models
MAX_TARGET_LENGTH = 256
RANDOM_SEED = 42

# Models to compare (pure text)
MODELS = {
    "biot5_sum": "QizhiPei/biot5-base",
    "t5_base_sum": "t5-base",
    "bart_base_sum": "facebook/bart-base",
    "biov2bart_sum": "GanjinZero/biobart-v2-base",
}

# Load JSONL
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

random.seed(RANDOM_SEED)
random.shuffle(data)

train_data, test_data = train_test_split(data, test_size=0.2, random_state=RANDOM_SEED)

# Save unseen test
with open(os.path.join(UNSEEN_PATH, "sum_unseen.jsonl"), "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

def tokenize_and_save(dataset_list, checkpoint, model_name, split):
    print(f"Tokenizing for {model_name} ({split})")

    # use_fast=False = more stable in Colab
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=False)

    ds = Dataset.from_list(dataset_list)

    def preprocess(example):
        src = example["input"]
        tgt = example["target"]

        # T5-family usually expects a task prefix
        if checkpoint in ["t5-base", "QizhiPei/biot5-base"]:
            src = "summarize: " + src

        model_inputs = tokenizer(
            src,
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding="max_length",
        )

        labels = tokenizer(
            text_target=tgt,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding="max_length",
        )

        # Mask target padding tokens so they are ignored in loss
        model_inputs["labels"] = [
            token if token != tokenizer.pad_token_id else -100
            for token in labels["input_ids"]
        ]

        return model_inputs

    tokenized = ds.map(preprocess, remove_columns=ds.column_names)

    out_dir = os.path.join(OUTPUT_DIR, model_name, split)

    # delete old folder to avoid schema mismatch
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    os.makedirs(out_dir, exist_ok=True)
    tokenized.save_to_disk(out_dir)

    print(f"Saved tokenized {split} to {out_dir}")

for model_name, ckpt in MODELS.items():
    tokenize_and_save(train_data, ckpt, model_name, "train")
    tokenize_and_save(test_data,  ckpt, model_name, "test")